# BaseLine

In [1]:
from hypex import ABTest
from hypex.dataset import (
    Dataset,
    FeatureRole,
    InfoRole,
    PreTargetRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils.tutorial_data_creation import DataGenerator

In [2]:
gen = DataGenerator(
    n_samples=1_000,
    distributions={
        "X1": {"type": "normal", "mean": 1, "std": 1},
        "X2": {"type": "bernoulli", "p": 0.5},
        "y0": {"type": "normal", "mean": 1, "std": 5},
    },
    time_correlations={"X1": 0.2, "X2": 0.1, "y0": 0.8},
    effect_size=0.1,
    seed=42
)

df = gen.generate()
# Keep only the columns we need for 2-period CUPAC
df = df.drop(columns=['y0', 'z', 'U', 'D', 'y1'])
df = df.rename(columns={'y0_lag_1': 'y_lag1', 'y0_lag_2': 'y_lag2'})

data = Dataset(
    roles = {
    "d": TreatmentRole(),
    "y": TargetRole(cofounders=["X1", "X2"]),

    "y_lag1": PreTargetRole(parent="y", lag=1),
    "X1_lag1": FeatureRole(parent="X1", lag=1),
    "X2_lag1": FeatureRole(parent="X2", lag=1),

    "y_lag2": PreTargetRole(parent="y", lag=2),
    "X1_lag2": FeatureRole(parent="X1", lag=2),
    "X2_lag2": FeatureRole(parent="X2", lag=2),
    },
    data=df,
    default_role=InfoRole(),
)

# Multilevel CUPAC with linear regression
test_cupac_linear = ABTest(
    enable_cupac=True,
)
result_cupac_linear = test_cupac_linear.execute(data)

result_cupac_linear.resume

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value
0,y,1,0.948518,1.590543,0.642025,67.687127,NOT OK,0.064382
1,y_cupac,1,1.007879,1.478836,0.470957,46.727496,OK,0.026723


In [3]:
result_cupac_linear.cupac.variance_reductions

,target,best_model,variance_reduction_cv,variance_reduction_real,control_mean_bias,test_mean_bias
0,y,ridge,66.0358,62.473053,-0.059361,0.111708


# Save and Load

## Save

In [ ]:
gen = DataGenerator(
    n_samples=1_000,
    distributions={
        "X1": {"type": "normal", "mean": 1, "std": 1},
        "X2": {"type": "bernoulli", "p": 0.5},
        "y0": {"type": "normal", "mean": 1, "std": 5},
    },
    time_correlations={"X1": 0.2, "X2": 0.1, "y0": 0.8},
    effect_size=0.1,
    seed=42
)

df = gen.generate()

df = df.drop(columns=['y0', 'z', 'U', 'D', 'y1', 'y'])
df = df.rename(columns={'y0_lag_1': 'y_lag1', 'y0_lag_2': 'y_lag2'})

data = Dataset(
    roles = {
    "d": TreatmentRole(),

    "y_lag1": PreTargetRole(parent="y", cofounders=["X1", "X2"], lag=1),
    "X1_lag1": FeatureRole(parent="X1", lag=1),
    "X2_lag1": FeatureRole(parent="X2", lag=1),

    "y_lag2": PreTargetRole(parent="y", lag=2),
    "X1_lag2": FeatureRole(parent="X1", lag=2),
    "X2_lag2": FeatureRole(parent="X2", lag=2),
    },
    data=df,
    default_role=InfoRole(),
)

test_cupac_linear = ABTest(
    enable_cupac=True,
    save_experiment=True
)
result_cupac_linear = test_cupac_linear.execute(data)

result_cupac_linear.resume

""


In [5]:
result_cupac_linear.cupac.variance_reductions

,target,best_model,variance_reduction_cv,variance_reduction_real,control_mean_bias,test_mean_bias
0,y,linear,66.003696,None,None,None


## Load

In [8]:
gen = DataGenerator(
    n_samples=1_000,
    distributions={
        "X1": {"type": "normal", "mean": 1, "std": 1},
        "X2": {"type": "bernoulli", "p": 0.5},
        "y0": {"type": "normal", "mean": 1, "std": 5},
    },
    time_correlations={"X1": 0.2, "X2": 0.1, "y0": 0.8},
    effect_size=0.1,
    seed=42
)

df = gen.generate()
# Keep only the columns we need for 2-period CUPAC
df = df.drop(columns=['y0', 'z', 'U', 'D', 'y1'])
df = df.rename(columns={'y0_lag_1': 'y_lag1', 'y0_lag_2': 'y_lag2'})

data = Dataset(
    roles = {
    "d": TreatmentRole(),
    "y": TargetRole(cofounders=["X1", "X2"]),

    "y_lag1": PreTargetRole(parent="y", lag=1),
    "X1_lag1": FeatureRole(parent="X1", lag=1),
    "X2_lag1": FeatureRole(parent="X2", lag=1),

    "y_lag2": PreTargetRole(parent="y", lag=2),
    "X1_lag2": FeatureRole(parent="X1", lag=2),
    "X2_lag2": FeatureRole(parent="X2", lag=2),
    },
    data=df,
    default_role=InfoRole(),
)

import os
exp_id = os.listdir(".hypex_experiments")[0] 

test_cupac_linear = ABTest(
    enable_cupac=True,
    load_experiment=exp_id,
)
result_cupac_linear = test_cupac_linear.execute(data)

result_cupac_linear.resume

ValueError: X has 3 features, but LinearRegression is expecting 1 features as input.

In [ ]:
result_cupac_linear.cupac.variance_reductions

,target,best_model,variance_reduction_cv,variance_reduction_real,control_mean_bias,test_mean_bias
0,y,ridge,66.0358,62.473053,-0.059361,0.111708
